In [54]:
import pandas as pd

DATA_PATH = "../data/raw/diabetic_data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

Dataset shape: (101766, 50)


In [55]:
print(df.shape)
print(df.columns.tolist())

(101766, 50)
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


In [56]:
# Create the 30-day readmission target

df["readmitted_30"] = (df["readmitted"] == "<30").astype(int)

TARGET = "readmitted_30"

ID_COLUMNS = [
    "encounter_id",
    "patient_nbr"
]

TIMING_SENSITIVE_COLUMNS = [
    "discharge_disposition_id"
]

print("Target:", TARGET)
print("ID columns:", ID_COLUMNS)
print("Timing-sensitive columns:", TIMING_SENSITIVE_COLUMNS)

Target: readmitted_30
ID columns: ['encounter_id', 'patient_nbr']
Timing-sensitive columns: ['discharge_disposition_id']


In [57]:
# Verify target column

if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' not found in dataset.")

print("Target column verified:", TARGET)
print("Target values:")
print(df[TARGET].value_counts(dropna=False))

Target column verified: readmitted_30
Target values:
readmitted_30
0    90409
1    11357
Name: count, dtype: int64


In [58]:
# Separate target from predictors

y = df[TARGET].copy()

X = df.drop(columns=[TARGET])

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target column:", TARGET)

X shape: (101766, 50)
y shape: (101766,)
Target column: readmitted_30


In [59]:
# Remove identifier columns from model predictors

X = X.drop(columns=ID_COLUMNS)

print("X shape after removing identifiers:", X.shape)
print("Removed columns:", ID_COLUMNS)

X shape after removing identifiers: (101766, 48)
Removed columns: ['encounter_id', 'patient_nbr']


In [60]:
# Remove the original readmission outcome from predictors

X = X.drop(columns=["readmitted"])

print("X shape after removing original target:", X.shape)
print("Is 'readmitted' still present?", "readmitted" in X.columns)

X shape after removing original target: (101766, 47)
Is 'readmitted' still present? False


In [61]:
# Review current predictor columns

print("Number of predictors:", len(X.columns))
print("\nPredictor columns:")
print(X.columns.tolist())

Number of predictors: 47

Predictor columns:
['race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


## Prediction Point

The model will predict the probability of 30-day hospital readmission at the time of patient discharge.

Therefore, features that are available at or by discharge can be considered as model inputs.

`discharge_disposition_id` is retained because the prediction point is defined at discharge. Its predictive usefulness and data quality will be evaluated during model development.

The final model will use only information that is legitimately available at the defined prediction point.

In [62]:
# Check actual NaN values in predictors

missing_values = X.isna().sum().sort_values(ascending=False)

print("Actual NaN values in predictors:")
print(missing_values[missing_values > 0])

Actual NaN values in predictors:
max_glu_serum    96420
A1Cresult        84748
dtype: int64


In [63]:
# Check '?' values in predictors

question_mark_counts = (
    X.astype(str)
     .apply(lambda col: (col == "?").sum())
     .sort_values(ascending=False)
)

print(" '?' values in predictors:")
print(question_mark_counts[question_mark_counts > 0])

 '?' values in predictors:
weight               98569
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
dtype: int64


In [64]:
# Create preprocessing copy

X_preprocessed = X.copy()

# Convert '?' to actual missing values
X_preprocessed = X_preprocessed.replace("?", pd.NA)

print("Preprocessing copy created.")
print("Original X shape:", X.shape)
print("Preprocessing copy shape:", X_preprocessed.shape)

Preprocessing copy created.
Original X shape: (101766, 47)
Preprocessing copy shape: (101766, 47)


In [65]:
# Verify that '?' values have been removed

question_mark_counts = (
    X_preprocessed.astype(str)
    .apply(lambda col: (col == "?").sum())
)

print("Remaining '?' values:")
print(question_mark_counts[question_mark_counts > 0])

Remaining '?' values:
Series([], dtype: int64)


In [66]:
# Review standardized missing values

missing_summary = pd.DataFrame({
    "missing_count": X_preprocessed.isna().sum(),
    "missing_percentage": X_preprocessed.isna().mean() * 100
})

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
].sort_values("missing_percentage", ascending=False)

print("Missing-value summary:")
print(missing_summary)

Missing-value summary:
                   missing_count  missing_percentage
weight                     98569           96.858479
max_glu_serum              96420           94.746772
A1Cresult                  84748           83.277322
medical_specialty          49949           49.082208
payer_code                 40256           39.557416
race                        2273            2.233555
diag_3                      1423            1.398306
diag_2                       358            0.351787
diag_1                        21            0.020636


In [67]:
# Classify predictor columns by data type

categorical_columns = X_preprocessed.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numerical_columns = X_preprocessed.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical columns:", len(categorical_columns))
print(categorical_columns)

print("\nNumerical columns:", len(numerical_columns))
print(numerical_columns)

Categorical columns: 36
['race', 'gender', 'age', 'weight', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']

Numerical columns: 11
['admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']


In [68]:
# Identify constant predictor columns

constant_columns = [
    col for col in X_preprocessed.columns
    if X_preprocessed[col].nunique(dropna=False) <= 1
]

print("Constant predictor columns:")
print(constant_columns)

Constant predictor columns:
['examide', 'citoglipton']


In [69]:
# Check categorical feature cardinality

cardinality = (
    X_preprocessed[categorical_columns]
    .nunique(dropna=True)
    .sort_values(ascending=False)
)

print("Categorical feature cardinality:")
print(cardinality)

Categorical feature cardinality:
diag_3                      789
diag_2                      748
diag_1                      716
medical_specialty            72
payer_code                   17
age                          10
weight                        9
race                          5
glyburide-metformin           4
metformin                     4
nateglinide                   4
repaglinide                   4
glipizide                     4
glyburide                     4
chlorpropamide                4
glimepiride                   4
miglitol                      4
acarbose                      4
rosiglitazone                 4
pioglitazone                  4
insulin                       4
gender                        3
max_glu_serum                 3
A1Cresult                     3
tolazamide                    3
acetohexamide                 2
tolbutamide                   2
troglitazone                  2
metformin-rosiglitazone       2
metformin-pioglitazone        2
glipizi

## Feature Preprocessing Decisions

### Constant features
- `examide` and `citoglipton` are constant and provide no variation.
- These features will be removed during preprocessing.

### High-cardinality categorical features
- `diag_1`, `diag_2`, and `diag_3` have hundreds of unique diagnosis codes.
- `medical_specialty` also has relatively high cardinality.
- These features should not be blindly one-hot encoded without considering dimensionality.
- The final encoding strategy will be determined during preprocessing/model development.

### Low-cardinality categorical features
- Demographic, medication, and diabetes-management categorical features can be encoded using an appropriate categorical encoding strategy.

### Missing values
- Missing categorical values will be handled explicitly.
- Features with very high missingness require separate consideration rather than automatic imputation.

In [70]:
constant_columns = ["examide", "citoglipton"]

X_preprocessed = X_preprocessed.drop(columns=constant_columns)

print("Removed:", constant_columns)
print("New shape:", X_preprocessed.shape)

Removed: ['examide', 'citoglipton']
New shape: (101766, 45)


In [71]:
# Remove features with extremely high missingness

high_missingness_columns = [
    "weight",
    "payer_code"
]

X_preprocessed = X_preprocessed.drop(
    columns=high_missingness_columns
)

print("Removed high-missingness columns:", high_missingness_columns)
print("New shape:", X_preprocessed.shape)

Removed high-missingness columns: ['weight', 'payer_code']
New shape: (101766, 43)


In [72]:
# Classify predictor columns for preprocessing

# These ID variables represent categories, not continuous numerical values
coded_categorical_columns = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

# Identify categorical columns stored as text
categorical_columns = X_preprocessed.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

# Add coded categorical variables
categorical_columns = categorical_columns + coded_categorical_columns

# Identify numerical columns
numerical_columns = [
    col
    for col in X_preprocessed.select_dtypes(
        include=["number"]
    ).columns
    if col not in coded_categorical_columns
]

print("Categorical columns:", len(categorical_columns))
print(categorical_columns)

print("\nNumerical columns:", len(numerical_columns))
print(numerical_columns)

print("\nTotal predictors:", len(X_preprocessed.columns))

Categorical columns: 35
['race', 'gender', 'age', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id']

Numerical columns: 8
['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Total predictors: 43


In [73]:
# Missing values by feature type

categorical_missing = (
    X_preprocessed[categorical_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

numerical_missing = (
    X_preprocessed[numerical_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Categorical features with missing values:")
print(categorical_missing[categorical_missing > 0])

print("\nNumerical features with missing values:")
print(numerical_missing[numerical_missing > 0])

Categorical features with missing values:
max_glu_serum        96420
A1Cresult            84748
medical_specialty    49949
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
dtype: int64

Numerical features with missing values:
Series([], dtype: int64)


## Missing-Value Strategy

Based on the missing-value analysis:

### Categorical features
Categorical missing values will be represented using an explicit `"Unknown"` category rather than being silently removed.

This applies to:
- `weight`
- `max_glu_serum`
- `A1Cresult`
- `medical_specialty`
- `payer_code`
- `race`
- `diag_1`
- `diag_2`
- `diag_3`

### Numerical features
No numerical predictors currently contain missing values, so numerical imputation is not required based on the current dataset.

### Important
Missing-value handling will be implemented inside the preprocessing pipeline so that preprocessing parameters are learned only from the training data.

In [74]:
# Review high-cardinality categorical features

cardinality = (
    X_preprocessed[categorical_columns]
    .nunique(dropna=True)
    .sort_values(ascending=False)
)

high_cardinality = cardinality[cardinality > 20]

print("High-cardinality categorical features:")
print(high_cardinality)

High-cardinality categorical features:
diag_3                      789
diag_2                      748
diag_1                      716
medical_specialty            72
discharge_disposition_id     26
dtype: int64


In [75]:
# Review low-cardinality categorical features

low_cardinality = cardinality[cardinality <= 20]

print("Low-cardinality categorical features:")
print(low_cardinality)

Low-cardinality categorical features:
admission_source_id         17
age                         10
admission_type_id            8
race                         5
glyburide-metformin          4
repaglinide                  4
pioglitazone                 4
glyburide                    4
glimepiride                  4
glipizide                    4
rosiglitazone                4
metformin                    4
chlorpropamide               4
nateglinide                  4
miglitol                     4
acarbose                     4
insulin                      4
gender                       3
tolazamide                   3
max_glu_serum                3
A1Cresult                    3
acetohexamide                2
tolbutamide                  2
troglitazone                 2
glipizide-metformin          2
glimepiride-pioglitazone     2
change                       2
metformin-pioglitazone       2
metformin-rosiglitazone      2
diabetesMed                  2
dtype: int64


## Categorical Encoding Strategy

### Low-cardinality categorical features
Low-cardinality categorical features will be handled using one-hot encoding.

### High-cardinality categorical features
The following features require separate handling because one-hot encoding could create a large number of columns:

- `diag_1`
- `diag_2`
- `diag_3`
- `medical_specialty`

No target-based encoding will be used because it could introduce target leakage if not carefully fitted within training folds.

The final treatment of these high-cardinality features will be implemented in a reproducible preprocessing pipeline.

In [76]:
# Review diagnosis code coverage

diagnosis_columns = ["diag_1", "diag_2", "diag_3"]

for col in diagnosis_columns:
    print(f"\n{col}")
    print("Unique values:", X_preprocessed[col].nunique(dropna=True))
    print("Missing values:", X_preprocessed[col].isna().sum())
    print("Top 10 values:")
    print(X_preprocessed[col].value_counts(dropna=False).head(10))


diag_1
Unique values: 716
Missing values: 21
Top 10 values:
diag_1
428    6862
414    6581
786    4016
410    3614
486    3508
427    2766
491    2275
715    2151
682    2042
434    2028
Name: count, dtype: int64

diag_2
Unique values: 748
Missing values: 358
Top 10 values:
diag_2
276    6752
428    6662
250    6071
427    5036
401    3736
496    3305
599    3288
403    2823
414    2650
411    2566
Name: count, dtype: int64

diag_3
Unique values: 789
Missing values: 1423
Top 10 values:
diag_3
250    11555
401     8289
276     5175
428     4577
427     3955
414     3664
496     2605
403     2357
585     1992
272     1969
Name: count, dtype: int64


## Diagnosis Feature Handling

The diagnosis features have high cardinality:

- `diag_1`: 716 unique codes
- `diag_2`: 748 unique codes
- `diag_3`: 789 unique codes

They also contain a relatively small number of missing values compared with their total size.

Because direct one-hot encoding would create a large number of sparse features, diagnosis columns will be handled separately from the low-cardinality categorical features.

The final diagnosis representation will be selected during model development and evaluated without using target information outside the training data.

In [77]:
# Review medical_specialty

col = "medical_specialty"

print("Unique values:", X_preprocessed[col].nunique(dropna=True))
print("Missing values:", X_preprocessed[col].isna().sum())

print("\nTop 15 values:")
print(X_preprocessed[col].value_counts(dropna=False).head(15))

Unique values: 72
Missing values: 49949

Top 15 values:
medical_specialty
NaN                                49949
InternalMedicine                   14635
Emergency/Trauma                    7565
Family/GeneralPractice              7440
Cardiology                          5352
Surgery-General                     3099
Nephrology                          1613
Orthopedics                         1400
Orthopedics-Reconstructive          1233
Radiologist                         1140
Pulmonology                          871
Psychiatry                           854
Urology                              685
ObstetricsandGynecology              671
Surgery-Cardiovascular/Thoracic      652
Name: count, dtype: int64


## Medical Specialty Handling

`medical_specialty` has 72 unique categories and 49,949 missing values.

Because of its relatively high cardinality and substantial missingness:

- Missing values will be represented explicitly as `"Unknown"`.
- The feature will be handled separately from the standard low-cardinality categorical features.
- It will not be dropped automatically based only on missingness.
- The final encoding approach will be implemented inside the preprocessing pipeline.

In [78]:
# Final feature groups for preprocessing

diagnosis_columns = [
    "diag_1",
    "diag_2",
    "diag_3"
]

specialty_columns = [
    "medical_specialty"
]

high_cardinality_columns = (
    diagnosis_columns + specialty_columns
)

low_cardinality_columns = [
    col
    for col in categorical_columns
    if col not in high_cardinality_columns
]

print("High-cardinality columns:")
print(high_cardinality_columns)

print("\nLow-cardinality columns:")
print(low_cardinality_columns)

print("\nNumerical columns:")
print(numerical_columns)

print(
    "\nTotal feature groups:",
    len(high_cardinality_columns)
    + len(low_cardinality_columns)
    + len(numerical_columns)
)

High-cardinality columns:
['diag_1', 'diag_2', 'diag_3', 'medical_specialty']

Low-cardinality columns:
['race', 'gender', 'age', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id']

Numerical columns:
['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Total feature groups: 43


## Preprocessing Pipeline Architecture

The preprocessing pipeline will use separate transformations for different feature groups.

### Numerical features

The numerical features include:

- `time_in_hospital`
- `num_lab_procedures`
- `num_procedures`
- `num_medications`
- `number_outpatient`
- `number_emergency`
- `number_inpatient`
- `number_diagnoses`

These features will be passed through the numerical preprocessing stage.

Scaling will be considered based on the selected model.

### Low-cardinality categorical features

Coded ID variables such as:

- `admission_type_id`
- `discharge_disposition_id`
- `admission_source_id`

will be treated as categorical variables rather than continuous numerical values.

Other low-cardinality categorical features will also be handled using categorical preprocessing.

Missing categorical values will be represented as `"Unknown"`.

One-hot encoding will be considered for low-cardinality categorical features.

### High-cardinality categorical features

The following features will be handled separately:

- `diag_1`
- `diag_2`
- `diag_3`
- `medical_specialty`

Missing values will be represented as `"Unknown"`.

Their encoding will be designed to avoid excessive dimensionality and target leakage.

### Leakage control

All preprocessing transformations that learn parameters from the data must be fitted only on the training data.

The preprocessing pipeline will therefore be integrated with the train/validation split rather than fitted on the complete dataset.